# Predicting heavy-equipment resale value

## Project goal

This notebook develops an interpretable model of **auction sale price** using historical heavy-equipment data from Kaggle's *Blue Book for Bulldozers* competition. The practical goal is to estimate an asset's residual value from its equipment class, age, meter hours, and sale year. That estimate can later serve as one input to a rent-versus-own or replacement calculator.

## Modeling approach

The analysis is intentionally iterative. It begins with a simple ordinary least squares (OLS) baseline, diagnoses weaknesses in the raw data, and adds complexity only when the results show that it is useful.

1. Load and narrow the source data.
2. Create age and market-year features.
3. Establish a simple price baseline.
4. Test log transformations and nonlinear age effects.
5. Investigate missing, zero, and physically impossible meter readings.
6. Replace broad product groups with detailed equipment classes.
7. Validate on the latest year rather than a random sample.
8. Measure reliability by equipment class and export the production artifacts.

> **Scope:** This is a residual-value model, not a repair-cost or maintenance-cost model. It estimates likely auction price from historical sales. The final estimate should be combined with operating, repair, financing, and rental costs before making an equipment decision.

## 1. Record the software environment

Model files can behave differently across library versions, especially when they are serialized for use in another application. Printing the Python and package versions creates a lightweight reproducibility record and makes deployment problems easier to diagnose later.

In [ ]:
import sys
import sklearn
import statsmodels
import joblib
import pandas
import numpy

print("Python:", sys.version)
print("scikit-learn:", sklearn.__version__)
print("statsmodels:", statsmodels.__version__)
print("joblib:", joblib.__version__)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
scikit-learn: 1.6.1
statsmodels: 0.15.0
joblib: 1.6.0
pandas: 2.2.3
numpy: 2.1.3


## 2. Load the auction data

The source file contains historical auction records for many types of heavy equipment. `saledate` is parsed as a date at import so calendar-year features can be calculated reliably. `low_memory=False` asks pandas to inspect complete columns before deciding their data types, which avoids mixed-type warnings on this wide file.

The preview is a quick structural check: it confirms that the download succeeded and that the expected fields are present before further work begins.

In [ ]:
import pandas as pd

url = "https://xplainable-public-storage.syd1.digitaloceanspaces.com/example_data/TrainAndValid.csv"

df = pd.read_csv(
    url,
    parse_dates=["saledate"],
    low_memory=False
)

df.head()

,SalesID,SalePrice,MachineID,ModelID,datasource,auctioneerID,YearMade,MachineHoursCurrentMeter,UsageBand,saledate,...,Undercarriage_Pad_Width,Stick_Length,Thumb,Pattern_Changer,Grouser_Type,Backhoe_Mounting,Blade_Type,Travel_Controls,Differential_Type,Steering_Controls
0,1139246,66000.0,999089,3157,121,3.0,2004,68.0,Low,2006-11-16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Standard,Conventional
1,1139248,57000.0,117657,77,121,3.0,1996,4640.0,Low,2004-03-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Standard,Conventional
2,1139249,10000.0,434808,7009,121,3.0,2001,2838.0,High,2004-02-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1139251,38500.0,1026470,332,121,3.0,2001,3486.0,High,2011-05-19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1139253,11000.0,1057373,17311,121,3.0,2007,722.0,Medium,2009-07-23,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Select the first modeling fields and engineer age

The initial table keeps only the target and a small set of intuitive predictors:

- **SalePrice:** the auction price to predict.
- **YearMade:** the machine's manufacturing year.
- **MachineHoursCurrentMeter:** cumulative recorded operating hours.
- **ProductGroup / ProductGroupDesc:** broad equipment type.
- **saledate:** the timing of the auction.

Two features are derived from the sale date. `sale_year` represents market conditions at the time of sale, while `equipment_age` represents physical age. Keeping those concepts separate matters: two identical ten-year-old machines may sell for different amounts in different market years.

In [ ]:
cols = [
    "SalePrice",
    "YearMade",
    "MachineHoursCurrentMeter",
    "saledate",
    "ProductGroup",
    "ProductGroupDesc"
]

equipment = df[cols].copy()

equipment["sale_year"] = equipment["saledate"].dt.year
equipment["equipment_age"] = (
    equipment["sale_year"] - equipment["YearMade"]
)

## 4. Define the initial modeling population

The first pass retains rows where meter hours are present. A regression cannot use a missing numeric predictor without an imputation strategy, and imputation would introduce an additional assumption before the quality of the meter field has been understood.

At this point, a recorded value of zero is still treated as observed data. The next diagnostics determine whether that is reasonable.

In [ ]:
model_df = equipment[
    equipment["MachineHoursCurrentMeter"].notna()
].copy()

## 5. Remove impossible manufacturing dates

Records are limited to equipment made in 1950 or later and to nonnegative calculated ages. These rules remove obvious coding problems such as placeholder manufacture years and machines that appear to have been sold before they were built.

This is a conservative cleaning step: it removes logically invalid rows without trimming legitimate old equipment simply because it is unusual.

In [ ]:
model_df = model_df[
    (model_df["YearMade"] >= 1950) &
    (model_df["equipment_age"] >= 0)
].copy()

### Initial distribution check

Descriptive statistics reveal the scale and shape of the three main numeric fields. The results show approximately **130,000 usable records**, a median sale price near **$27,000**, and a median equipment age of **7 years**. Meter hours are much more problematic: the median is only 399, the 25th percentile is zero, and the maximum exceeds 2.4 million hours.

Those meter results are the first indication that zeros may represent unknown values and that some positive readings may also be invalid. They motivate the more detailed quality checks later in the notebook.

In [ ]:
model_df[
    [
        "SalePrice",
        "equipment_age",
        "MachineHoursCurrentMeter"
    ]
].describe()

,SalePrice,equipment_age,MachineHoursCurrentMeter
count,130521.000000,130521.000000,1.305210e+05
mean,34824.763909,9.899005,3.408614e+03
std,24961.143493,7.670566,2.662534e+04
min,4750.000000,0.000000,0.000000e+00
25%,16000.000000,5.000000,0.000000e+00
50%,27000.000000,7.000000,3.990000e+02
75%,46000.000000,13.000000,3.266000e+03
max,142000.000000,60.000000,2.483300e+06


## 6. Establish a deliberately simple baseline

The first OLS model predicts sale price directly from age and raw meter hours. Its purpose is not to be the final model; it provides a benchmark for judging whether later features genuinely improve explanatory power.

The low R² (about **0.05**) shows that age and raw meter hours alone explain very little of the price variation. The positive hours coefficient is also counterintuitive. That does not necessarily mean additional use increases value—it is a warning that equipment size/type and poor meter data are confounding the relationship.

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols(
    """
    SalePrice ~
    equipment_age +
    MachineHoursCurrentMeter
    """,
    data=model_df
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              SalePrice   R-squared:                       0.048
Model:                            OLS   Adj. R-squared:                  0.048
Method:                 Least Squares   F-statistic:                     3311.
Date:                Fri, 18 Sep 2026   Prob (F-statistic):               0.00
Time:                        15:29:01   Log-Likelihood:            -1.5035e+06
No. Observations:              130521   AIC:                         3.007e+06
Df Residuals:                  130518   BIC:                         3.007e+06
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

## 7. Control for broad equipment type

Equipment types operate on very different price scales, so `C(ProductGroup)` adds a separate categorical adjustment for each broad group. This prevents the model from treating a skid steer and a motor grader as directly comparable simply because their ages and hours are similar.

R² rises from roughly **0.05 to 0.40**, demonstrating that equipment identity is far more important than age or hours alone. The large improvement also suggests that an even more detailed class variable may be valuable later.

In [ ]:
model_type = smf.ols(
    """
    SalePrice ~
    equipment_age +
    MachineHoursCurrentMeter +
    C(ProductGroup)
    """,
    data=model_df
).fit()

print(model_type.summary())

                            OLS Regression Results                            
Dep. Variable:              SalePrice   R-squared:                       0.402
Model:                            OLS   Adj. R-squared:                  0.402
Method:                 Least Squares   F-statistic:                 1.253e+04
Date:                Fri, 18 Sep 2026   Prob (F-statistic):               0.00
Time:                        15:29:02   Log-Likelihood:            -1.4732e+06
No. Observations:              130521   AIC:                         2.946e+06
Df Residuals:                  130513   BIC:                         2.946e+06
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

## 8. Model proportional rather than absolute price differences

Auction prices are right-skewed, and a $10,000 error has a very different meaning for a $15,000 machine than for a $150,000 machine. Applying `log1p` to sale price:

- reduces the influence of the highest-priced equipment,
- makes the residual spread more consistent across price levels, and
- lets effects be interpreted approximately as percentage changes rather than fixed dollar changes.

The log-price model reaches an R² of about **0.49**, an improvement over modeling raw dollars with the same predictors. Predictions will later be converted back to dollars with `expm1`.

In [ ]:
import numpy as np

model_df["log_sale_price"] = np.log1p(
    model_df["SalePrice"]
)

log_model = smf.ols(
    """
    log_sale_price ~
    equipment_age +
    MachineHoursCurrentMeter +
    C(ProductGroup)
    """,
    data=model_df
).fit()

print(log_model.summary())

                            OLS Regression Results                            
Dep. Variable:         log_sale_price   R-squared:                       0.491
Model:                            OLS   Adj. R-squared:                  0.491
Method:                 Least Squares   F-statistic:                 1.796e+04
Date:                Fri, 18 Sep 2026   Prob (F-statistic):               0.00
Time:                        15:29:02   Log-Likelihood:                -95586.
No. Observations:              130521   AIC:                         1.912e+05
Df Residuals:                  130513   BIC:                         1.913e+05
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

### Rescale meter hours for readability

Dividing operating hours by 1,000 changes only the units, not the information or model fit. A coefficient on `machine_hours_1000` describes the effect of an additional 1,000 hours, which is easier to read than a coefficient for one hour. The following refit confirms that rescaling does not alter the model's explanatory power.

In [ ]:
model_df["machine_hours_1000"] = (
    model_df["MachineHoursCurrentMeter"] / 1000
)

In [ ]:
log_model = smf.ols(
    """
    log_sale_price ~
    equipment_age +
    machine_hours_1000 +
    C(ProductGroup)
    """,
    data=model_df
).fit()

print(log_model.summary())

                            OLS Regression Results                            
Dep. Variable:         log_sale_price   R-squared:                       0.491
Model:                            OLS   Adj. R-squared:                  0.491
Method:                 Least Squares   F-statistic:                 1.796e+04
Date:                Fri, 18 Sep 2026   Prob (F-statistic):               0.00
Time:                        15:29:03   Log-Likelihood:                -95586.
No. Observations:              130521   AIC:                         1.912e+05
Df Residuals:                  130513   BIC:                         1.913e+05
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 10

## 9. Diagnose the meter-hours field

Before relying on meter hours, the notebook examines its percentiles, the share of exact zeros, group-level distributions, and correlations. Nearly **48% of the initial records have a zero meter reading**, making it unlikely that every zero represents a genuinely unused machine.

The working interpretation is that many zeros are missing or unavailable readings encoded as `0`. Because that issue can obscure the true relationship between use and resale value, the analysis compares models with and without those rows rather than silently accepting them.

In [ ]:
model_df["MachineHoursCurrentMeter"].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]
)

,MachineHoursCurrentMeter
count,1.305210e+05
mean,3.408614e+03
std,2.662534e+04
min,0.000000e+00
1%,0.000000e+00
5%,0.000000e+00
25%,0.000000e+00
50%,3.990000e+02
75%,3.266000e+03
95%,1.031300e+04


In [ ]:
(model_df["MachineHoursCurrentMeter"] == 0).mean()

np.float64(0.4769883773492388)

In [ ]:
model_df.groupby("ProductGroup")[
    "MachineHoursCurrentMeter"
].describe()

,count,mean,std,min,25%,50%,75%,max
ProductGroup,,,,,,,,
BL,24916.0,2532.767860,21362.593040,0.0,0.0,8.0,2580.00,932000.0
MG,7177.0,2864.812596,19741.876651,0.0,0.0,0.0,3073.00,1194900.0
SSL,16142.0,2288.886012,29255.292989,0.0,0.0,888.5,1833.00,2483300.0
TEX,33546.0,4171.807160,29781.180624,0.0,0.0,1297.0,4555.75,2202400.0
TTT,26194.0,3295.567916,26109.451676,0.0,0.0,0.0,3240.00,1711700.0
WL,22546.0,4347.099086,27343.659002,0.0,0.0,0.0,4945.00,1728600.0


In [ ]:
model_df[
    ["equipment_age", "MachineHoursCurrentMeter", "log_sale_price"]
].corr()

,equipment_age,MachineHoursCurrentMeter,log_sale_price
equipment_age,1.000000,0.006149,-0.222297
MachineHoursCurrentMeter,0.006149,1.000000,0.016660
log_sale_price,-0.222297,0.016660,1.000000


## 10. Allow depreciation to curve with age

Straight-line depreciation assumes each additional year changes log price by the same amount. In practice, equipment often loses value quickly when newer and more slowly once it is older. Adding `age_squared` allows one smooth curve to represent that changing rate.

Adjusted R² increases only modestly—from about **0.491 to 0.496**—but the comparison shows that the nonlinear term contains useful information. Both age terms must be interpreted together; the positive squared term does not mean old equipment appreciates on its own.

In [ ]:
model_df["age_squared"] = model_df["equipment_age"] ** 2

model_age_curve = smf.ols(
    """
    log_sale_price ~
    equipment_age +
    age_squared +
    machine_hours_1000 +
    C(ProductGroup)
    """,
    data=model_df
).fit()

print(model_age_curve.summary())

                            OLS Regression Results                            
Dep. Variable:         log_sale_price   R-squared:                       0.496
Model:                            OLS   Adj. R-squared:                  0.496
Method:                 Least Squares   F-statistic:                 1.607e+04
Date:                Fri, 18 Sep 2026   Prob (F-statistic):               0.00
Time:                        15:29:03   Log-Likelihood:                -94869.
No. Observations:              130521   AIC:                         1.898e+05
Df Residuals:                  130512   BIC:                         1.898e+05
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 10

In [ ]:
print("Linear age:", log_model.rsquared_adj)
print("Curved age:", model_age_curve.rsquared_adj)

Linear age: 0.4905933077741783
Curved age: 0.49615536625879686


## 11. Isolate machines with reported use

Rows with meter hours greater than zero are placed in a separate dataset. This reduces the sample from about **130,500 to 68,300 records**, but it avoids interpreting likely missing-value placeholders as truly unused equipment.

The adjusted R² changes only slightly at this stage. The value of this filter is therefore mainly data validity and coefficient interpretability, not an immediate jump in fit.

In [ ]:
hours_df = model_df[
    model_df["MachineHoursCurrentMeter"] > 0
].copy()

hours_df["machine_hours_1000"] = (
    hours_df["MachineHoursCurrentMeter"] / 1000
)

hours_df["age_squared"] = (
    hours_df["equipment_age"] ** 2
)

In [ ]:
hours_model = smf.ols(
    """
    log_sale_price ~
    equipment_age +
    age_squared +
    machine_hours_1000 +
    C(ProductGroup)
    """,
    data=hours_df
).fit()

print(hours_model.summary())

                            OLS Regression Results                            
Dep. Variable:         log_sale_price   R-squared:                       0.499
Model:                            OLS   Adj. R-squared:                  0.499
Method:                 Least Squares   F-statistic:                     8484.
Date:                Fri, 18 Sep 2026   Prob (F-statistic):               0.00
Time:                        15:29:04   Log-Likelihood:                -51232.
No. Observations:               68264   AIC:                         1.025e+05
Df Residuals:                   68255   BIC:                         1.026e+05
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 10

In [ ]:
print("Original n:", len(model_df))
print("Known-hours n:", len(hours_df))
print("Original adj R²:", model_age_curve.rsquared_adj)
print("Known-hours adj R²:", hours_model.rsquared_adj)

Original n: 130521
Known-hours n: 68264
Original adj R²: 0.49615536625879686
Known-hours adj R²: 0.49852920755374375


In [ ]:
hours_df["MachineHoursCurrentMeter"].describe(
    percentiles=[.01, .05, .25, .50, .75, .95, .99]
)

,MachineHoursCurrentMeter
count,6.826400e+04
mean,6.517281e+03
std,3.654018e+04
min,1.000000e+00
1%,5.863000e+01
5%,4.430000e+02
25%,1.553000e+03
50%,3.067000e+03
75%,6.080250e+03
95%,1.371970e+04


## 12. Apply a physical plausibility check

A machine cannot accumulate more than 8,760 operating hours per year—the number of calendar hours in a year. `max_possible_hours` uses `(age + 1)` so equipment sold during its manufacture year can still have up to one year of use.

Only **380 records** violate this generous upper bound. They are inspected before removal so the rule is transparent and so valid high-use equipment is not confused with impossible readings.

In [ ]:
hours_df["max_possible_hours"] = (
    (hours_df["equipment_age"] + 1) * 8760
)

hours_df["invalid_hours"] = (
    hours_df["MachineHoursCurrentMeter"]
    > hours_df["max_possible_hours"]
)

In [ ]:
hours_df["invalid_hours"].value_counts()

,count
invalid_hours,
False,67884
True,380


In [ ]:
hours_df.loc[
    hours_df["invalid_hours"],
    [
        "equipment_age",
        "MachineHoursCurrentMeter",
        "ProductGroup",
        "SalePrice"
    ]
].sort_values(
    "MachineHoursCurrentMeter",
    ascending=False
).head(30)

,equipment_age,MachineHoursCurrentMeter,ProductGroup,SalePrice
324615,4,2483300.0,SSL,8500.0
347409,13,2202400.0,TEX,12000.0
324899,4,1857100.0,SSL,12000.0
297550,14,1729600.0,TEX,22000.0
309643,17,1728600.0,WL,27000.0
309549,11,1711700.0,TTT,80000.0
327710,7,1602900.0,WL,37500.0
301601,14,1485900.0,TTT,70000.0
305333,10,1282700.0,TEX,40000.0
348423,15,1228200.0,TEX,18500.0


### Create the cleaned known-hours sample

The impossible readings are removed, leaving **67,884 observations** with positive and physically possible meter hours. The percentile table that follows is another reasonableness check on the cleaned range; it is not an additional trimming rule.

In [ ]:
clean_hours_df = hours_df[
    ~hours_df["invalid_hours"]
].copy()

In [ ]:
clean_hours_df[
    "MachineHoursCurrentMeter"
].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99, .995]
)

,MachineHoursCurrentMeter
count,67884.000000
mean,4549.371089
std,4870.526094
min,1.000000
1%,57.000000
5%,439.000000
25%,1547.000000
50%,3043.000000
75%,6010.000000
95%,13147.850000


## 13. Test a log transformation of operating hours

Meter readings remain strongly right-skewed even after invalid values are removed. `log1p(hours)` compresses the long upper tail and represents diminishing marginal wear: the difference between 500 and 1,500 hours may carry more information than the difference between 20,500 and 21,500 hours.

The raw-hours and log-hours models perform almost identically (adjusted R² about **0.5402 versus 0.5405**). The log version is retained because it is slightly stronger and less sensitive to extreme readings, not because the improvement is dramatic.

In [ ]:
clean_hours_df["log_machine_hours"] = np.log1p(
    clean_hours_df["MachineHoursCurrentMeter"]
)

In [ ]:
model_raw_hours = smf.ols(
    """
    log_sale_price ~
    equipment_age +
    age_squared +
    machine_hours_1000 +
    C(ProductGroup)
    """,
    data=clean_hours_df
).fit()

model_log_hours = smf.ols(
    """
    log_sale_price ~
    equipment_age +
    age_squared +
    log_machine_hours +
    C(ProductGroup)
    """,
    data=clean_hours_df
).fit()

print("Raw hours:", model_raw_hours.rsquared_adj)
print("Log hours:", model_log_hours.rsquared_adj)

Raw hours: 0.5401665439840656
Log hours: 0.5405348687101796


### Review the most heavily used machines

`hours_per_year` provides context for extreme but still technically possible meter readings. Listing the 20 largest observations is a manual diagnostic: it helps distinguish data errors from machines that plausibly operated for multiple shifts or in high-utilization environments.

No additional rows are removed here. This avoids tailoring the dataset solely to make the regression cleaner.

In [ ]:
clean_hours_df["hours_per_year"] = (
    clean_hours_df["MachineHoursCurrentMeter"]
    / (clean_hours_df["equipment_age"] + 1)
)

clean_hours_df.nlargest(
    20,
    "MachineHoursCurrentMeter"
)[
    [
        "ProductGroup",
        "equipment_age",
        "MachineHoursCurrentMeter",
        "hours_per_year",
        "SalePrice"
    ]
]

,ProductGroup,equipment_age,MachineHoursCurrentMeter,hours_per_year,SalePrice
326954,TTT,30,161800.0,5219.354839,9500.0
349508,TTT,17,120000.0,6666.666667,10250.0
356867,TTT,18,120000.0,6315.789474,8000.0
325707,MG,23,96900.0,4037.500000,13000.0
380741,BL,14,96890.0,6459.333333,21000.0
365299,WL,33,94075.0,2766.911765,16500.0
304842,TEX,14,89300.0,5953.333333,26000.0
380535,TEX,13,78765.0,5626.071429,40000.0
102732,WL,21,78740.0,3579.090909,38000.0
327145,MG,25,76500.0,2942.307692,10000.0


### Refit the transformed exploratory model

This model combines the decisions made so far: log sale price, curved age, log meter hours, and broad product group. Its summary is useful for checking coefficient directions, significance, residual diagnostics, and overall fit after cleaning.

The duplicate import and feature statements make this section runnable after the cleaned dataset exists, even if the earlier exploratory cells are rearranged. The following raw-hours refit is retained as a direct sensitivity comparison.

In [ ]:
import numpy as np
import statsmodels.formula.api as smf

clean_hours_df["log_machine_hours"] = np.log1p(
    clean_hours_df["MachineHoursCurrentMeter"]
)

clean_hours_df["age_squared"] = (
    clean_hours_df["equipment_age"] ** 2
)

model_log_hours = smf.ols(
    """
    log_sale_price ~
    equipment_age +
    age_squared +
    log_machine_hours +
    C(ProductGroup)
    """,
    data=clean_hours_df
).fit()

print(model_log_hours.summary())

                            OLS Regression Results                            
Dep. Variable:         log_sale_price   R-squared:                       0.541
Model:                            OLS   Adj. R-squared:                  0.541
Method:                 Least Squares   F-statistic:                     9984.
Date:                Fri, 18 Sep 2026   Prob (F-statistic):               0.00
Time:                        15:29:05   Log-Likelihood:                -47960.
No. Observations:               67884   AIC:                         9.594e+04
Df Residuals:                   67875   BIC:                         9.602e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  9

In [ ]:
model_raw_hours = smf.ols(
    """
    log_sale_price ~
    equipment_age +
    age_squared +
    MachineHoursCurrentMeter +
    C(ProductGroup)
    """,
    data=clean_hours_df
).fit()

print("Raw hours:", model_raw_hours.rsquared_adj)
print("Log hours:", model_log_hours.rsquared_adj)

Raw hours: 0.5401665439840656
Log hours: 0.5405348687101796


## 14. Replace broad groups with detailed equipment classes

Broad groups still combine machines with very different capacity and value. `fiProductClassDesc` captures distinctions such as horsepower, operating capacity, digging depth, and tonnage. `ProductSize` is copied for inspection, while the class description becomes the principal categorical predictor.

The fields are joined back by the original dataframe index, preserving alignment with the already-cleaned rows. `sale_year` is also reconstructed from the source date for market adjustment and temporal validation.

In [ ]:
clean_hours_df["fiProductClassDesc"] = (
    df.loc[clean_hours_df.index, "fiProductClassDesc"]
)

In [ ]:
clean_hours_df[
    ["fiProductClassDesc", "ProductSize"]
] = df.loc[
    clean_hours_df.index,
    ["fiProductClassDesc", "ProductSize"]
]

In [ ]:
clean_hours_df["sale_year"] = pd.to_datetime(
    df.loc[clean_hours_df.index, "saledate"]
).dt.year

In [ ]:
clean_hours_df[
    ["fiProductClassDesc", "ProductSize", "sale_year"]
].head()

,fiProductClassDesc,ProductSize,sale_year
0,Wheel Loader - 110.0 to 120.0 Horsepower,NaN,2006
1,Wheel Loader - 150.0 to 175.0 Horsepower,Medium,2004
2,Skid Steer Loader - 1351.0 to 1601.0 Lb Operat...,NaN,2004
3,"Hydraulic Excavator, Track - 12.0 to 14.0 Metr...",Small,2011
4,Skid Steer Loader - 1601.0 to 1751.0 Lb Operat...,NaN,2009


## 15. Compare class detail and market timing

The first detailed model treats sale year as a categorical effect, allowing each auction year to have its own market adjustment. It reaches an adjusted R² of about **0.841**, a major improvement over the broad-group model.

The next fit temporarily excludes year to show how much market timing contributes. The stepwise comparison then adds class, market year, age, and hours in sequence:

- detailed class alone: **0.646** adjusted R²;
- adding market year: **0.658**;
- adding age and curved age: **0.841**;
- adding meter hours: **0.841**.

The key finding is that detailed class and age explain most of the model's performance. Meter hours add relatively little after those fields are known, although they remain operationally meaningful.

In [ ]:
class_year_model = smf.ols(
    """
    log_sale_price ~
    equipment_age +
    age_squared +
    log_machine_hours +
    C(fiProductClassDesc) +
    C(sale_year)
    """,
    data=clean_hours_df
).fit()

print(class_year_model.summary())

                            OLS Regression Results                            
Dep. Variable:         log_sale_price   R-squared:                       0.841
Model:                            OLS   Adj. R-squared:                  0.841
Method:                 Least Squares   F-statistic:                     3865.
Date:                Fri, 18 Sep 2026   Prob (F-statistic):               0.00
Time:                        15:29:09   Log-Likelihood:                -11877.
No. Observations:               67884   AIC:                         2.394e+04
Df Residuals:                   67790   BIC:                         2.480e+04
Df Model:                          93                                         
Covariance Type:            nonrobust                                         
                                                                                          coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------

In [ ]:
class_model = smf.ols(
    """
    log_sale_price ~
    equipment_age +
    age_squared +
    log_machine_hours +
    C(fiProductClassDesc)
    """,
    data=clean_hours_df
).fit()

print(class_model.summary())
print("Adjusted R²:", class_model.rsquared_adj)

                            OLS Regression Results                            
Dep. Variable:         log_sale_price   R-squared:                       0.822
Model:                            OLS   Adj. R-squared:                  0.822
Method:                 Least Squares   F-statistic:                     4298.
Date:                Fri, 18 Sep 2026   Prob (F-statistic):               0.00
Time:                        15:29:10   Log-Likelihood:                -15720.
No. Observations:               67884   AIC:                         3.159e+04
Df Residuals:                   67810   BIC:                         3.226e+04
Df Model:                          73                                         
Covariance Type:            nonrobust                                         
                                                                                          coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------

In [ ]:
m1 = smf.ols(
    "log_sale_price ~ C(fiProductClassDesc)",
    data=clean_hours_df
).fit()

m2 = smf.ols(
    """
    log_sale_price ~
    C(fiProductClassDesc) +
    C(sale_year)
    """,
    data=clean_hours_df
).fit()

m3 = smf.ols(
    """
    log_sale_price ~
    C(fiProductClassDesc) +
    C(sale_year) +
    equipment_age +
    age_squared
    """,
    data=clean_hours_df
).fit()

m4 = smf.ols(
    """
    log_sale_price ~
    C(fiProductClassDesc) +
    C(sale_year) +
    equipment_age +
    age_squared +
    log_machine_hours
    """,
    data=clean_hours_df
).fit()

print("Class only:           ", m1.rsquared_adj)
print("Class + market year:  ", m2.rsquared_adj)
print("+ equipment age:      ", m3.rsquared_adj)
print("+ operating hours:    ", m4.rsquared_adj)

Class only:            0.6459222824709576
Class + market year:   0.6575600996210602
+ equipment age:       0.8407689449801892
+ operating hours:     0.8411062424895032


## 16. Validate on future data

A random split would let auctions from every year appear in both training and test data, which can make performance look better than a real deployment. Instead, all sales before 2012 form the training set and 2012 is held out as an unseen future period.

The validation model uses `sale_year` as a numeric trend rather than a set of year categories. This is necessary because a categorical model trained only through 2011 would not have learned a coefficient for the unseen 2012 category. The split is repeated in the next cell before fitting; this duplication is harmless and keeps the modeling block self-contained.

In [ ]:
train = clean_hours_df[
    clean_hours_df["sale_year"] < 2012
].copy()

test = clean_hours_df[
    clean_hours_df["sale_year"] == 2012
].copy()

In [ ]:
train = clean_hours_df[
    clean_hours_df["sale_year"] < 2012
].copy()

test = clean_hours_df[
    clean_hours_df["sale_year"] == 2012
].copy()

temporal_model = smf.ols(
    """
    log_sale_price ~
    C(fiProductClassDesc) +
    sale_year +
    equipment_age +
    age_squared +
    log_machine_hours
    """,
    data=train
).fit()

pred = temporal_model.predict(test)

## 17. Evaluate accuracy in both model space and dollars

No single metric tells the full story:

- **Log R²** measures how much variation the model explains on the scale it was trained on.
- **Dollar MAE** reports the average absolute dollar miss after predictions are converted back with `expm1`.
- **Median absolute percentage error** describes the typical proportional miss and is less dominated by a few expensive machines.
- **Median signed percentage error** checks systematic direction. A negative value means the model tends to underpredict.

On the 2012 holdout, the model achieves **0.766 log R²**, about **$10,290 MAE**, and roughly **23.0% median absolute percentage error**. The median signed error is about **−16.2%**, so the uncorrected predictions tend to be conservative. That bias should be surfaced in any calculator that uses the model.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

print(
    "Test log R²:",
    r2_score(test["log_sale_price"], pred)
)

pred_dollars = np.expm1(pred)

print(
    "Dollar MAE:",
    mean_absolute_error(
        test["SalePrice"],
        pred_dollars
    )
)

Test log R²: 0.7662067856795658
Dollar MAE: 10289.804884505882


In [ ]:
test["predicted_price"] = pred_dollars

test["absolute_pct_error"] = (
    abs(test["SalePrice"] - test["predicted_price"])
    / test["SalePrice"]
)

print(
    "Median absolute % error:",
    test["absolute_pct_error"].median()
)

Median absolute % error: 0.22995773155736776


In [ ]:
test["pct_error"] = (
    test["predicted_price"] - test["SalePrice"]
) / test["SalePrice"]

print(
    "Median signed % error:",
    test["pct_error"].median()
)

Median signed % error: -0.16224009086966168


## 18. Measure reliability by equipment class

Overall metrics can hide weak performance for rare classes. The holdout results are therefore grouped by detailed equipment class to report:

- the number of 2012 test observations,
- the median actual auction price, and
- the median absolute percentage error.

The table is sorted from lowest to highest error. Because a small sample can produce an unstable median, the next view requires at least 20 test observations before a class is described as reliable.

In [ ]:
class_accuracy = (
    test.groupby("fiProductClassDesc")
    .agg(
        count=("SalePrice", "size"),
        median_actual_price=("SalePrice", "median"),
        median_abs_pct_error=("absolute_pct_error", "median")
    )
    .sort_values("median_abs_pct_error")
)

class_accuracy

,count,median_actual_price,median_abs_pct_error
fiProductClassDesc,,,
Wheel Loader - Unidentified,3,13000.0,0.067233
Wheel Loader - 60.0 to 80.0 Horsepower,76,14000.0,0.112098
Backhoe Loader - Unidentified,9,13500.0,0.130369
"Track Type Tractor, Dozer - 20.0 to 75.0 Horsepower",116,19000.0,0.165646
"Track Type Tractor, Dozer - 75.0 to 85.0 Horsepower",70,23000.0,0.169290
...,...,...,...
"Track Type Tractor, Dozer - 190.0 to 260.0 Horsepower",23,53000.0,0.430168
Motorgrader - 45.0 to 130.0 Horsepower,34,21050.0,0.436444
"Hydraulic Excavator, Track - 11.0 to 12.0 Metric Tons",5,27500.0,0.456676


In [ ]:
reliable_classes = class_accuracy[
    class_accuracy["count"] >= 20
].copy()

reliable_classes.sort_values("median_abs_pct_error")

,count,median_actual_price,median_abs_pct_error
fiProductClassDesc,,,
Wheel Loader - 60.0 to 80.0 Horsepower,76,14000.0,0.112098
"Track Type Tractor, Dozer - 20.0 to 75.0 Horsepower",116,19000.0,0.165646
"Track Type Tractor, Dozer - 75.0 to 85.0 Horsepower",70,23000.0,0.169290
"Hydraulic Excavator, Track - 3.0 to 4.0 Metric Tons",99,15000.0,0.170072
"Hydraulic Excavator, Track - 0.0 to 2.0 Metric Tons",35,10000.0,0.172312
Skid Steer Loader - 2701.0+ Lb Operating Capacity,47,14000.0,0.181291
"Track Type Tractor, Dozer - 160.0 to 190.0 Horsepower",68,57250.0,0.184143
Backhoe Loader - 16.0 + Ft Standard Digging Depth,40,19000.0,0.186151
"Hydraulic Excavator, Track - 4.0 to 5.0 Metric Tons",30,20250.0,0.186236


### Create user-facing confidence labels

The confidence function combines sample size and observed holdout error:

- **High:** at least 50 test sales and median error no greater than 25%.
- **Moderate:** at least 20 test sales and median error no greater than 40%.
- **Low:** all other combinations.

These are pragmatic communication thresholds, not formal prediction intervals. They help a calculator warn users when a point estimate is based on sparse or historically inconsistent data. Across classes with at least 20 observations, the overall median absolute percentage error remains about **22.9%**.

In [ ]:
def confidence_level(row):
    if row["count"] >= 50 and row["median_abs_pct_error"] <= 0.25:
        return "High"
    elif row["count"] >= 20 and row["median_abs_pct_error"] <= 0.40:
        return "Moderate"
    else:
        return "Low"

class_accuracy["confidence"] = class_accuracy.apply(
    confidence_level,
    axis=1
)

In [ ]:
reliable_test = test[
    test["fiProductClassDesc"].isin(
        reliable_classes.index
    )
]

print(
    "Reliable-class median APE:",
    reliable_test["absolute_pct_error"].median()
)

Reliable-class median APE: 0.2290143701367423


## 19. Refit the deployment model on all cleaned data

After model design and validation are complete, the same formula is fit once more using the full cleaned dataset, including 2012. This is intentional: the holdout remains untouched while choices are evaluated, then its information is recovered for the artifact that will power the calculator.

The deployed formula uses detailed class, numeric sale year, curved age, and log meter hours to predict log sale price. A future application must recreate these transformations exactly and should reject equipment classes that were not present during training.

In [ ]:
calculator_model = smf.ols(
    """
    log_sale_price ~
    C(fiProductClassDesc) +
    sale_year +
    equipment_age +
    age_squared +
    log_machine_hours
    """,
    data=clean_hours_df
).fit()

## 20. Export the model and allowed classes

The remaining cells create the files needed by a downstream application:

1. Save the fitted statsmodels result as a pickle.
2. Export the sorted class list for a valid dropdown menu.
3. Test a compressed `joblib` alternative and measure its file size.
4. Re-save the statsmodels model with `remove_data=True` to reduce unnecessary training-data payload.

The exported model is still approximately **48 MiB**, so repository or hosting file-size limits should be checked. Serialization also ties the artifact to compatible Python and statsmodels versions—the environment record at the top of the notebook is therefore part of the deployment documentation.

In [ ]:
from pathlib import Path

Path("models").mkdir(exist_ok=True)

calculator_model.save(
    "models/residual_value_model.pkl"
)

In [ ]:
classes = sorted(
    clean_hours_df["fiProductClassDesc"]
    .dropna()
    .unique()
)

pd.Series(classes).to_csv(
    "models/equipment_classes.csv",
    index=False,
    header=False
)

In [ ]:
import joblib
from pathlib import Path

Path("models").mkdir(exist_ok=True)

joblib.dump(
    calculator_model,
    "models/residual_value_model.joblib",
    compress=3
)

['models/residual_value_model.joblib']

In [ ]:
from pathlib import Path

size_mb = Path("models/residual_value_model.joblib").stat().st_size / (1024 ** 2)
print(f"{size_mb:.1f} MB")

44.4 MB


In [ ]:
calculator_model.save(
    "models/residual_value_model.pkl",
    remove_data=True
)

In [ ]:
from pathlib import Path

model_path = Path("models/residual_value_model.pkl")

size_bytes = model_path.stat().st_size
size_mib = size_bytes / (1024 ** 2)

print(f"Model size: {size_mib:.2f} MiB")

Model size: 47.91 MiB


## Interpretation and limitations

The final model is useful as an **estimated historical auction value**, but it is not a complete ownership decision by itself.

- The data end in 2012, so dollar predictions are not automatically adjusted to today's market or inflation.
- Rows with missing or zero meter readings are excluded; predictions may not generalize to machines without reliable hours.
- Condition, geography, manufacturer, model, attachments, and auction-specific factors are not included in the final formula.
- The observed 2012 bias suggests that the raw point estimates tend to be low.
- Confidence labels summarize historical test performance; they are not statistical prediction intervals or guarantees.

For a rent-versus-own calculator, the residual-value estimate should be shown with a range and combined with acquisition price, expected utilization, rental rates, transportation, maintenance, financing, downtime, and the planned ownership horizon. A strong next modeling step would be to add inflation/current-market calibration and generate class-specific prediction intervals.